In [17]:
from src.labels import *
from src.models import *
from src.preprocessing import *
from src.splits import *
from src.train import *

In [29]:
import pandas as pd

# Show every row
pd.set_option('display.max_rows', None)

# Show every column
pd.set_option('display.max_columns', None)

# Show full content of each cell (no truncation or "...")
pd.set_option('display.max_colwidth', None)

# Prevent long frames from wrapping lines across pages
pd.set_option('display.expand_frame_repr', False)

In [18]:
data = pd.read_csv("data/parsed_save2.csv")

In [19]:
pre = preflop_data(data)
pos = position_mapping(pre)
classified = action_label(pos)
temp_with_id = classified.reset_index()
train_set, test_set = split_data_with_id_hash(temp_with_id, test_ratio=0.2, id_column="index")
x_train, y_train = x_y_separator(train_set)

In [20]:
x_train.drop(columns=["Unnamed: 0","if_fold", "result"], inplace=True)


In [21]:
x_train = hand_label(x_train)

In [22]:
x_train.drop(columns=['index','stage_flop',
       'stage_turn', 'stage_river',
       'board_r1', 'board_r2', 'board_r3',
       'board_r4', 'board_r5', 'hero_bet_ratio'], inplace=True)

In [24]:
comparison, fitted_searches = compare_models(x_train, y_train)

In [30]:
print(comparison)

           model  cv_macro_f1  cv_macro_f1_std  train_macro_f1  balanced_accuracy                                                                           best_parameters
2  decision_tree     0.569046         0.081435        0.755036           0.620218  {'model__class_weight': 'balanced', 'model__max_depth': 6, 'model__min_samples_leaf': 1}
1       logistic     0.548927         0.084639        0.607032           0.540335                                            {'model__C': 100, 'model__class_weight': None}
0          dummy     0.167829         0.000949        0.167832           0.250000                                                                                        {}


In [31]:
print(fitted_searches)

{'dummy': GridSearchCV(cv=RepeatedStratifiedKFold(n_repeats=5, n_splits=5, random_state=42),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('position',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         ['position_norm']),
                                                                        ('numerical',
                                                                         StandardScaler(),
                                                                         ['hero_r1',
                                                                          'hero_r2',
                      